# AIDev – the main dataset
**Paper:** Li et al. 2025 – AIDev (used by Rahman, Sawada, Who Finishes, Khemissi, Oukay, Khelifi, Dovhoshliubnyi)
**Data:** https://huggingface.co/datasets/hao-li/AIDev

**What is in the data:** Pull requests written by AI agents on GitHub (Codex, Copilot, Devin, Cursor, Claude Code, Jules), with repositories, commits, reviews and the **code changes** (`pr_commit_details`). `pull_request` = AIDev-pop (repos with >100 stars).

**One row of the main table =** one AI pull request

This notebook has 3 parts:
1. **First look** – the files, some lines, all the columns, the size.
2. **Data-science view** – size, dates, **numbers per AI agent**, **how the agent is involved** (tables saved in `tables/`).
3. **Charts** – what kind of change · fixes per AI agent · who wrote the fix · how fast the fix comes · extra (saved in `charts/`).

In [ ]:
import sys, os, json
sys.path.insert(0, "../..")          # to find common.py in the parent folder
from common import *

## 0. Get the data
⚠️ AIDev is big (≈3 GB). If you already downloaded it in **evolution-assurance** (`data/raw`), put its path below instead of downloading again. The version on Hugging Face today is **v4** (up to Nov 2025).

In [ ]:
import duckdb
RAW = r"C:\Users\GIGABYTE\Desktop\code_theis\evolution-assurance\data\raw"   # <- change if needed
if not os.path.exists(RAW):
    from huggingface_hub import hf_hub_download
    RAW = "data/aidev"; os.makedirs(RAW, exist_ok=True)
    for f in ["pull_request.parquet", "all_pull_request.parquet", "repository.parquet", "pr_commits.parquet", "pr_commit_details.parquet"]:
        hf_hub_download("hao-li/AIDev", f, repo_type="dataset", local_dir=RAW)
P_ = lambda f: os.path.join(RAW, f).replace("\\", "/")
con = duckdb.connect()
sorted(os.listdir(RAW))

## 1. First look

In [ ]:
pr = con.execute(f"SELECT * FROM '{P_('pull_request.parquet')}'").df()
_ = overview(pr.drop(columns=[c for c in ["body"] if c in pr]), "AIDev-pop pull requests")
repo = con.execute(f"SELECT * FROM '{P_('repository.parquet')}'").df()
_ = overview(repo, "Repositories")

In [ ]:
commits = con.execute(f"SELECT * FROM '{P_('pr_commits.parquet')}' LIMIT 2000").df()
_ = overview(commits, "Commits (first 2,000)")
details = con.execute(f"SELECT * FROM '{P_('pr_commit_details.parquet')}' LIMIT 2000").df()
_ = overview(details.drop(columns=[c for c in ["patch"] if c in details]), "Changed files (first 2,000)")

## 2. Data-science view
### Size and dates (whole AIDev and AIDev-pop)

In [ ]:
for name in ["all_pull_request.parquet", "pull_request.parquet"]:
    print(name); display(con.execute(f"""SELECT COUNT(*) n_prs, COUNT(DISTINCT repo_id) n_repos, SUM(merged_at IS NOT NULL) n_merged,
        MIN(TRY_CAST(created_at AS TIMESTAMP)) first_pr, MAX(TRY_CAST(created_at AS TIMESTAMP)) latest_pr FROM '{P_(name)}'""").df())

### Numbers per AI agent (+ Python only)

In [ ]:
per_agent = con.execute(f"""SELECT agent, COUNT(*) prs, SUM(merged_at IS NOT NULL) merged,
    ROUND(100.0*SUM(merged_at IS NOT NULL)/COUNT(*),1) merge_pct, MAX(TRY_CAST(created_at AS TIMESTAMP)) latest_pr
    FROM '{P_('all_pull_request.parquet')}' GROUP BY agent ORDER BY prs DESC""").df()
py = pr.merge(repo[["id", "language"]].rename(columns={"id": "repo_id"}), on="repo_id")
per_agent = per_agent.merge(py[py.language == "Python"].groupby("agent").size().rename("python_prs_pop").reset_index(), on="agent", how="left")
save_table(per_agent, "per_agent")
per_agent

### How the agent is involved
The agent **opens the PR** (`agent`, `user`). Inside the PR, commits can be made by the agent or by humans – see the author columns of the commits table above.

In [ ]:
summary_card("AIDev (v4)", "AI pull request", int(per_agent.prs.sum()), int(repo.shape[0]), str(pr.created_at.min())[:10],
             str(per_agent.latest_pr.max())[:10], ", ".join(per_agent.agent), "Agent opens the PR; commits by agent and/or humans", "need linker.py")

## 3. Charts

In [ ]:
bar(per_agent.set_index("agent")["prs"], "AI PRs per agent (all AIDev)", "1_prs_per_agent.png", horizontal=True)
stacked(per_agent.set_index("agent")[["merged"]].assign(not_merged=per_agent.set_index("agent")["prs"] - per_agent.set_index("agent")["merged"]).iloc[::-1], "Merged vs not merged, per agent", "2_merged_per_agent.png", colors=[BLUE, LIGHT])
bar(py.language.value_counts().head(10), "AIDev-pop PRs per repository language", "3_languages.png", horizontal=True)

**Fixes / who fixes / how fast:** AIDev has no fix links by itself – that is exactly what `linker.py` (Who Finishes) builds. See folder **01_WhoFinishes** for those 4 charts.